# Example XX Using the PRMS prep meteorology Module

In [1]:
from pathlib import Path
import os

import geopandas as gpd
import numpy as np
import xarray as xr
import rioxarray
from shapely.geometry import Polygon
from chmdata.thredds import GridMet, BBox

In [2]:
src_root = (Path('C:/Users/CNB968/OneDrive - MT/GitHub/MIHMS') / 'src').resolve()
os.chdir(src_root)

import mihms.prep.prms as pprms
from mihms.prep.utils import load_raster_window

In [3]:
out_pth = Path(os.getcwd()).parent / 'example/Yellowstone'

elev_pth = Path(r'D:\Spatial_Data\Statewide_Data\Raster\SRTM_Elevation\MT_hydro_SRTM_30m.tif')
basin_pth = Path(r'D:\ArcGIS_Projects\Yellowstone\Upper Yellowstone\Vector\UYBoundary_abv_Shields_5071.shp')
model_grid_pth = Path('C:/Users/CNB968/OneDrive - MT/GitHub/MIHMS/example/Yellowstone/UY_voronoi_grid.grid')
param_fl_pth = Path('C:/Users/CNB968/OneDrive - MT/GitHub/MIHMS/example/Yellowstone/UY_static_params.param')

basin = gpd.read_file(basin_pth)

In [4]:
basin_bounds = basin.to_crs(4326).bounds
met_bbox = BBox(basin_bounds.loc[0,'minx'], basin_bounds.loc[0,'maxx'], basin_bounds.loc[0,'maxy'], basin_bounds.loc[0,'miny'])
p = GridMet('pr', start='2020-01-01', end='2024-12-31', bbox=met_bbox).subset_nc(return_array=True)
tmin = GridMet('tmmn', start='2020-01-01', end='2024-12-31', bbox=met_bbox).subset_nc(return_array=True)
tmax = GridMet('tmmx', start='2020-01-01', end='2024-12-31', bbox=met_bbox).subset_nc(return_array=True)
met = xr.merge([p, tmin, tmax])

In [6]:
met['daily_maximum_temperature'].values = met.daily_maximum_temperature - 273.15
met['daily_minimum_temperature'].values = met.daily_minimum_temperature - 273.15

In [5]:
bnds = met.rio.bounds()
met_clip = gpd.GeoDataFrame(geometry=[Polygon([(bnds[0], bnds[1]), (bnds[0], bnds[3]), (bnds[2], bnds[3]), (bnds[2], bnds[1])])], crs=met.rio.crs)
met_elev = load_raster_window(met_clip, elev_pth, output='xarray')
met_elev.name = 'elevation'
nan_msk = np.where(met_elev.values[0] == 0, np.nan, met_elev.values[0])
new_elev = np.array([nan_msk])
met_elev.values = new_elev

Geometry and raster CRS do not match, reprojecting geometry to match raster dataset...


In [7]:
met_params = pprms.PRMSParameters(grid=model_grid_pth)
met_params, met_data = pprms.temp_1sta_from_gridded(met,met_elev, param_obj=met_params, station_limit=100)

params.py:134: UserWarning: There are no parameters specified yet and or no nhru dimension to check grid. Assigning grid anyway, make sure to check any inserted parameters match the grid size.


In [ ]:
prdf = met_data.data_df.loc[:,met_data.data_df.columns.str.contains(pat='precip')]


,precip_0,precip_1,precip_2,precip_3,precip_4,precip_5,precip_6,precip_7,precip_8,precip_9,...,precip_94,precip_95,precip_96,precip_97,precip_98,precip_99,precip_100,precip_101,precip_102,precip_103
0,9.488889,5.500000,0.400000,0.000000,0.000000,11.788889,7.255556,0.822222,5.911111,1.533333,...,8.811111,11.611111,11.855556,12.977778,14.433332,11.388889,16.133333,16.444445,16.233334,11.355556
1,0.244444,0.177778,0.000000,0.000000,0.000000,0.055556,0.133333,0.000000,0.033333,0.000000,...,2.577778,2.288889,2.388889,2.200000,2.822222,3.233333,2.444444,2.611111,3.533334,3.533334
2,0.300000,0.177778,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,4.822222,2.477778,2.588889,2.522223,2.877778,3.777778,2.455556,2.066667,2.177778,2.188889
3,0.344444,0.188889,0.000000,0.077778,0.366667,0.066667,0.000000,0.000000,0.344444,0.300000,...,1.022222,4.666667,4.333333,3.977778,4.244444,3.122222,4.900000,5.177778,5.888889,4.444444
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.033333,0.000000,0.000000,0.000000,0.000000,...,2.144444,1.544444,1.800000,2.000000,2.322222,1.844445,2.677778,2.600000,1.922222,0.866667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1822,0.322222,0.244444,0.000000,0.033333,0.100000,0.000000,0.000000,0.000000,1.066667,0.833333,...,3.311111,7.388889,6.488889,5.544445,4.388889,2.988889,4.977778,3.866667,2.600000,1.933333
1823,1.388889,1.388889,0.044444,0.000000,0.000000,1.066667,0.866667,0.155556,2.744445,1.283333,...,11.522222,10.066668,10.477777,11.022222,10.733334,11.000000,10.888888,9.777779,8.677777,9.855556
1824,9.155556,10.044445,3.800000,3.744445,4.966667,8.822223,8.022222,4.944445,13.200000,10.750000,...,23.022223,24.244446,24.311110,24.433334,23.311111,20.533333,22.822222,20.622221,17.944445,17.722225
1825,8.433332,7.311112,0.744444,0.000000,0.050000,10.755555,6.433333,1.044445,4.433333,2.316667,...,3.577778,1.088889,1.666667,2.333333,2.677778,2.555556,2.377778,2.211111,1.955556,1.666667


In [6]:
basin_bounds = basin.to_crs(4326).bounds
met_bbox = BBox(basin_bounds.loc[0,'minx'], basin_bounds.loc[0,'maxx'], basin_bounds.loc[0,'maxy'], basin_bounds.loc[0,'miny'])
p = GridMet('pr', start='1979-01-01', end='2024-12-31', bbox=met_bbox).subset_nc(return_array=True)
tmin = GridMet('tmmn', start='1979-01-01', end='2024-12-31', bbox=met_bbox).subset_nc(return_array=True)
tmax = GridMet('tmmx', start='1979-01-01', end='2024-12-31', bbox=met_bbox).subset_nc(return_array=True)
met = xr.merge([p, tmin, tmax])
met['daily_maximum_temperature'].values = met.daily_maximum_temperature - 273.15
met['daily_minimum_temperature'].values = met.daily_minimum_temperature - 273.15

In [7]:
param_fl = pprms.PRMSParameters.load_paramfile(param_fl_pth)

------------------------------------
Reading parameter file : UY_static_params.param
------------------------------------


In [8]:
param_fl.parameters

<xarray.Dataset>
Dimensions:               (one: 1, nhru: 45244, nsub: 19, nsegment: 1352,
                           ncascade: 88141, nlake: 1, nmonths: 12, ntemp: 104,
                           nssr: 45244, ngw: 45244, ndeplval: 11)
Dimensions without coordinates: one, nhru, nsub, nsegment, ncascade, nlake,
                                nmonths, ntemp, nssr, ngw, ndeplval
Data variables: (12/80)
    elev_units            (one) int32 1
    hru_type              (nhru) int32 1 1 1 1 1 1 1 1 1 1 ... 1 1 1 1 1 1 1 1 1
    hru_subbasin          (nhru) int32 19 19 19 19 19 19 19 ... 17 16 1 17 1 15
    hru_lat               (nhru) float64 45.75 45.74 45.74 ... 45.11 45.11 44.81
    hru_lon               (nhru) float64 -110.7 -110.6 -110.6 ... -110.2 -110.8
    lake_hru_id           (nhru) int32 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0
    ...                    ...
    den_max               (nhru) float64 0.6 0.6 0.6 0.6 0.6 ... 0.6 0.6 0.6 0.6
    emis_noppt            (nhru) float64 0.757 0.757 0.757 ... 0.757 0.757 0.757
    melt_force            (nhru) int32 140 140 140 140 140 ... 140 140 140 140
    melt_look             (nhru) int32 90 90 90 90 90 90 ... 90 90 90 90 90 90
    cecn_coef             (nmonths, nhru) float64 5.0 5.0 5.0 ... 5.0 5.0 5.0
    tstorm_mo             (nmonths, nhru) int32 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0

In [8]:
met_params, met_data = pprms.temp_1sta_from_gridded(met,met_elev, model_grid=model_grid_pth, param_obj=param_fl, station_limit=100)

params.py:360: UserWarning: The parameter already exists, overwriting the existing 'basin_tsta' parameter.
params.py:360: UserWarning: The parameter already exists, overwriting the existing 'hru_psta' parameter.
params.py:360: UserWarning: The parameter already exists, overwriting the existing 'hru_tsta' parameter.
params.py:360: UserWarning: The parameter already exists, overwriting the existing 'max_missing' parameter.
params.py:360: UserWarning: The parameter already exists, overwriting the existing 'rain_adj' parameter.
params.py:360: UserWarning: The parameter already exists, overwriting the existing 'snow_adj' parameter.
params.py:360: UserWarning: The parameter already exists, overwriting the existing 'tmax_adj' parameter.
params.py:360: UserWarning: The parameter already exists, overwriting the existing 'tmin_adj' parameter.
params.py:360: UserWarning: The parameter already exists, overwriting the existing 'temp_units' parameter.
params.py:360: UserWarning: The parameter alread

In [9]:
met_params.parameters

<xarray.Dataset>
Dimensions:               (one: 1, nhru: 45244, nsub: 19, nsegment: 1352,
                           ncascade: 88141, nlake: 1, nmonths: 12, ntemp: 104,
                           nssr: 45244, ngw: 45244, ndeplval: 11, nrain: 104)
Coordinates:
  * nrain                 (nrain) int32 0 1 2 3 4 5 6 ... 98 99 100 101 102 103
Dimensions without coordinates: one, nhru, nsub, nsegment, ncascade, nlake,
                                nmonths, ntemp, nssr, ngw, ndeplval
Data variables: (12/83)
    elev_units            (one) int32 1
    hru_type              (nhru) int32 1 1 1 1 1 1 1 1 1 1 ... 1 1 1 1 1 1 1 1 1
    hru_subbasin          (nhru) int32 19 19 19 19 19 19 19 ... 17 16 1 17 1 15
    hru_lat               (nhru) float64 45.75 45.74 45.74 ... 45.11 45.11 44.81
    hru_lon               (nhru) float64 -110.7 -110.6 -110.6 ... -110.2 -110.8
    lake_hru_id           (nhru) int32 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0
    ...                    ...
    melt_look             (nhru) int32 90 90 90 90 90 90 ... 90 90 90 90 90 90
    cecn_coef             (nmonths, nhru) float64 5.0 5.0 5.0 ... 5.0 5.0 5.0
    tstorm_mo             (nmonths, nhru) int32 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0
    tmax_allrain_offset   (nmonths, nhru) float64 0.0 0.0 0.0 ... 0.0 0.0 0.0
    tmax_allsnow          (nmonths, nhru) float64 0.0 0.0 0.0 ... 0.0 0.0 0.0
    adjmix_rain           (nmonths, nhru) float64 1.0 1.0 1.0 ... 1.0 1.0 1.0

In [10]:
met_params.remove_parameters(['tmax_allrain_offset', 'tmax_allsnow', 'adjmix_rain'])

In [11]:
met_params.parameters

<xarray.Dataset>
Dimensions:               (one: 1, nhru: 45244, nsub: 19, nsegment: 1352,
                           ncascade: 88141, nlake: 1, nmonths: 12, ntemp: 104,
                           nssr: 45244, ngw: 45244, ndeplval: 11, nrain: 104)
Coordinates:
  * nrain                 (nrain) int32 0 1 2 3 4 5 6 ... 98 99 100 101 102 103
Dimensions without coordinates: one, nhru, nsub, nsegment, ncascade, nlake,
                                nmonths, ntemp, nssr, ngw, ndeplval
Data variables: (12/80)
    elev_units            (one) int32 1
    hru_type              (nhru) int32 1 1 1 1 1 1 1 1 1 1 ... 1 1 1 1 1 1 1 1 1
    hru_subbasin          (nhru) int32 19 19 19 19 19 19 19 ... 17 16 1 17 1 15
    hru_lat               (nhru) float64 45.75 45.74 45.74 ... 45.11 45.11 44.81
    hru_lon               (nhru) float64 -110.7 -110.6 -110.6 ... -110.2 -110.8
    lake_hru_id           (nhru) int32 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0
    ...                    ...
    den_max               (nhru) float64 0.6 0.6 0.6 0.6 0.6 ... 0.6 0.6 0.6 0.6
    emis_noppt            (nhru) float64 0.757 0.757 0.757 ... 0.757 0.757 0.757
    melt_force            (nhru) int32 140 140 140 140 140 ... 140 140 140 140
    melt_look             (nhru) int32 90 90 90 90 90 90 ... 90 90 90 90 90 90
    cecn_coef             (nmonths, nhru) float64 5.0 5.0 5.0 ... 5.0 5.0 5.0
    tstorm_mo             (nmonths, nhru) int32 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0

In [12]:
met_params.pygsflow_param_obj.write()